# Tutorial 1 — Prepare brain IDPs

**Goal:** create label-aligned predictor (`X`) and outcome (`Y`) tables. The external input is a BN region-by-IDP table; HomoloMap supplies the cell-type predictors. This notebook performs only data preparation and quality control.

<!-- github-visual-preview -->
![Tutorial visual preview](figures/01_prepare_brain_idps.png)

*Deterministic preview generated from the released BN map and example IDPs. Running the notebook redraws the figure from the current inputs.*


In [ ]:
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

HERE = Path.cwd() if (Path.cwd() / 'tutorial_utils.py').exists() else Path.cwd() / 'tutorials'
sys.path.insert(0, str(HERE))
from tutorial_utils import find_repo_root, load_celltype_map, load_idps, align_and_validate

ROOT = find_repo_root()
CELLTYPE_LEVEL = 'subclass'  # or 'cluster'
IDP_PATH = None  # e.g. ROOT / 'my_data' / 'brain_idps_bn.csv'

## Input contract

The IDP CSV must have BN region labels as rows and imaging phenotypes as columns. Labels—not row position—define alignment. The fallback creates deterministic toy IDPs only to verify execution.

In [ ]:
X_source = load_celltype_map(CELLTYPE_LEVEL, ROOT)
Y_source, is_toy = load_idps(IDP_PATH, X_source.index, seed=42)
X, Y = align_and_validate(X_source, Y_source)

print('Toy IDPs:', is_toy)
print('Cell types:', X.shape, 'IDPs:', Y.shape)
print('BN labels:', X.index.min(), 'to', X.index.max())
display(Y.head())

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3.5))
sns.heatmap(X.T, cmap='mako', xticklabels=False, ax=ax)
ax.set(xlabel='BN regions', ylabel='Cell types', title='Aligned cell-type composition')
fig.tight_layout()

## Save validated inputs

Later tutorials load these files. Record the atlas, hemisphere, feature resolution, retained ROI labels, and mapping audit with a scientific analysis.

In [ ]:
OUTPUT = ROOT / 'tutorial_outputs'
OUTPUT.mkdir(exist_ok=True)
X.to_csv(OUTPUT / 'aligned_celltype_predictors.csv')
Y.to_csv(OUTPUT / 'aligned_brain_idps.csv')
print('Saved to', OUTPUT.resolve())

<!-- tutorial-visual-summary -->
### Visual quality control
The upper panel shows the supplied brain IDPs across ordered BN labels; the lower panel confirms compositional closure.


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(10, 5), sharex=True,
                         gridspec_kw={'height_ratios': [3, 1]})
Y.plot(ax=axes[0], linewidth=1.5)
axes[0].set(ylabel='IDP value', title='Brain IDPs in BN regional order')
axes[0].legend(frameon=False, ncol=min(4, Y.shape[1]))
axes[1].plot(X.index, X.sum(axis=1), color='#2a9d8f', linewidth=1.5)
axes[1].axhline(1, color='0.25', linestyle='--', linewidth=0.8)
axes[1].set(xlabel='BN region label', ylabel='Row sum', ylim=(0.98, 1.02))
sns.despine()
fig.tight_layout()
